# Day 4: The Black Box Recorder. See What Really Happened

**Today's goal:** make every SupportDesk request leave a trail, so that when an answer is bad you can open the trail and say *exactly which step failed*, how long each step took, and what it cost. All without storing anyone's personal data.

**Why it matters:** *Team Orion* shipped a refund agent. At 3:14 AM finance asked why $40,000 went out, and nobody could answer, because there was no trace of what the agent saw and decided. Today you build the thing they were missing.

## The big analogy: the flight recorder and the parcel tracking number

When a plane has a problem, investigators do not guess. They open the **black box**: a record of every moment, with timings. And when you order a parcel, every scan carries the **same tracking number**, so you can follow *your* parcel among millions.

| In aviation / delivery | In our agent | Today's word |
| --- | --- | --- |
| The black box log | A file with one JSON line per step | **Trace** (made of **spans**) |
| One scan of the parcel | One record: *who, what, how long, did it fail* | **Span** |
| The tracking number on every scan | One id per request, carried through every step | **Correlation id (`cid`)** |
| "Handled by the sorting centre" vs "handled by the truck" | Three kinds of span: our own code, a tool, the model | **node / tool / model** |
| Fuel used on the flight | Tokens and cost of each model call | **Tokens, cost** |
| "Package changed from *packed* to *shipped*" | Which state boxes a step changed | **State transition** |
| The recorder never stores the passengers' conversations | Only sizes, counts and names are logged, never the question | **Privacy** |
| The investigator's report | Symptom, root cause, evidence | **Incident note** |

**The one idea to remember:** the symptom shows up in the *last* step (a bad answer), but the cause is often an *earlier* one. Only a trace lets you tell them apart.

## How to use this notebook
1. Run each cell from top to bottom: click a cell, then press **Shift + Enter**.
2. Read the short story above each cell first.
3. Nothing here calls a real AI model. A small stub gives the same *shape* of answer (with token counts) as Ollama. See the note in Step 2 if you want a real call.
4. The notebook creates `day04_trace.jsonl` next to itself and deletes it at the end.
5. In **Step 8** you will *predict* before you run.
6. Correlation ids and millisecond values change on every run. That is normal.


## Step 0: Setup

- `json` writes the spans, `time` measures them, `uuid` makes tracking numbers.
- `TRACE_FILE` is the black box.
- `FAULT` lets us break the balance tool on purpose (failure injection).
- `TOOL` holds how long the balance tool takes (we will change it in an experiment).
- `PRICE_PER_1K_TOKENS` is **illustrative**. It is not a real price list. Use your provider's prices in real life.

In [1]:
import json
import operator
import os
import re
import time
import uuid
from typing import Annotated, TypedDict

from langgraph.graph import END, START, StateGraph

TRACE_FILE = "day04_trace.jsonl"
USE_OLLAMA = False   # False = offline stub with Ollama-shaped token counts; True = real call (needs `ollama signin`)
PRICE_PER_1K_TOKENS = {"in": 0.0005, "out": 0.0015}   # illustrative USD prices, not a real price list
FAULT = {"lookup_balance": False}                       # failure injection: True makes the balance tool time out
TOOL = {"delay_s": 0.05}                                # how long the balance tool takes
POLICY_WORDS = {"policy", "fee", "fees", "charge", "rule", "loan", "prepayment", "lost", "stolen", "report"}
P1 = "P1: Lost or stolen cards must be reported within 24 hours. A frozen card is unfrozen at a branch."

if os.path.exists(TRACE_FILE):
    os.remove(TRACE_FILE)   # start with an empty black box
print("Setup done. Black box is empty.")

Setup done. Black box is empty.


## Step 1: The case file and the one-line log: `tr()`

### The state
The same SupportDesk case file as Day 3. Two boxes matter today:
- `cid`: the **tracking number** of this request, made once at the front door.
- `t0`: the **start time** of the request.

### The `tr()` line
Every step writes one short, human-readable line into `trace`, in this exact format (the same as the Day 19 capstone):

```text
cid=<tracking number> <step name> +<milliseconds since the request started> <detail>
```

The `+ms` is **cumulative from `t0`**, so you can compare steps with each other.

In [2]:
class State(TypedDict):   # the SupportDesk state, same names as Day 19
    user: str
    team: str
    question: str
    cid: str
    t0: float
    route: str
    context: list
    tool: str
    result: str
    answer: str
    tokens: int
    blocked: str
    fallback: bool
    trace: Annotated[list, operator.add]   # every step appends one line (Day 3's rule)


def tr(state, node, detail=""):
    """One trace line (identical in Day 19): correlation id, node name, cumulative milliseconds, detail."""
    return f"cid={state['cid']} {node} +{int((time.perf_counter() - state['t0']) * 1000)}ms {detail}".strip()

print(tr({"cid": "demo1234", "t0": time.perf_counter()}, "decide_route", "route=tool"))

cid=demo1234 decide_route +0ms route=tool


## Step 2: The recorder: spans

A **span** is one scan of the parcel: one JSON line in the file. Every span has:

| Field | Meaning |
| --- | --- |
| `cid` | The tracking number |
| `kind` | `node` (our code), `tool` (an outside tool) or `model` (the AI) |
| `name` | Which one |
| `latency_ms` | How long it took |
| `error` | `null` if fine, otherwise the error text |

Model spans also have `tokens_in`, `tokens_out` and `cost_usd`. Node spans have `state_keys_changed`.

We build four small helpers. Read the **wrapper idea** closely, because it is the heart of today:

- `emit(...)` is the **only** place that writes to the file. It stores sizes, counts and names. **Never the question or the answer.**
- `traced(name, node)` wraps a step like a **stopwatch and a camera**: it times the step, catches any error, writes the span, and lets the graph carry on. The steps themselves contain **no tracing code**.
- `call_tool(...)` records the tool span. It records the error and then **raises it again** so the node span also sees the failure. That is why one failure shows up as two error lines (the tool, and the node that called it).
- `call_model(...)` records tokens and cost. By default it uses a **stub** that returns Ollama-shaped counts, so you need no login. To try a real model, set `USE_OLLAMA = True` after `ollama signin` (a real answer can contain odd characters, so keep printing simple).

In [3]:
def emit(cid, kind, name, start, error=None, **fields):
    """Append one JSON span. Only sizes, counts and names go in: never the raw question or answer."""
    record = {"cid": cid, "kind": kind, "name": name, "latency_ms": round((time.perf_counter() - start) * 1000, 1), "error": error}
    record.update(fields)
    with open(TRACE_FILE, "a") as trace:
        trace.write(json.dumps(record) + "\n")


def traced(name, node):   # node span: latency, error, and which state keys the node changed
    def wrapper(state):
        start = time.perf_counter()
        error, update = None, {}
        try:
            update = node(state)
        except Exception as exc:   # error-as-data: record it, degrade, let the graph continue
            error = type(exc).__name__ + ": " + str(exc)
            update = {"fallback": True, "result": "", "trace": [tr(state, name, "ERROR " + type(exc).__name__)]}
        emit(state["cid"], "node", name, start, error, state_keys_changed=sorted(update))
        return update
    return wrapper


def call_tool(cid, name, tool, *args):   # tool span, separate from the node that called it
    start = time.perf_counter()
    error = None
    try:
        return tool(*args)
    except Exception as exc:
        error = type(exc).__name__ + ": " + str(exc)
        raise   # the node span records the same failure; the tool span tells us WHICH tool
    finally:
        emit(cid, "tool", name, start, error)


def call_model(cid, prompt):   # model span: token counts and cost
    start = time.perf_counter()
    if USE_OLLAMA:
        import ollama
        response = ollama.chat(model="gpt-oss:120b-cloud", messages=[{"role": "user", "content": prompt}])
    else:
        response = {"message": {"content": "Here is what I found."}, "prompt_eval_count": len(prompt) // 4, "eval_count": 9}
    tokens_in, tokens_out = response["prompt_eval_count"], response["eval_count"]
    cost = (tokens_in * PRICE_PER_1K_TOKENS["in"] + tokens_out * PRICE_PER_1K_TOKENS["out"]) / 1000
    emit(cid, "model", "gpt-oss:120b-cloud", start, tokens_in=tokens_in, tokens_out=tokens_out, cost_usd=round(cost, 6))
    return response["message"]["content"], tokens_in + tokens_out

print("Recorder helpers ready: emit, traced, call_tool, call_model.")

Recorder helpers ready: emit, traced, call_tool, call_model.


## Step 3: The workers and the tool

Five small workers, as on Day 3. Look at what they do **not** contain: no timing, no logging code. They only add their one `tr()` line. The stopwatch comes from `traced`.

`lookup_balance` is a **stand-in** for the real MCP tool you build on Days 6 to 8. It waits a moment, then either answers or (if we flip `FAULT`) raises a `TimeoutError`. Note that its answer contains a card number. That is on purpose: we will check later that the number **never reaches the trace**.

In [4]:
def lookup_balance(user):   # stand-in for the MCP tool (Days 6-8)
    time.sleep(TOOL["delay_s"])
    if FAULT["lookup_balance"]:
        raise TimeoutError("mcp server did not answer in " + str(TOOL["delay_s"]) + "s")
    return "Balance for " + user + " is 1,250 INR (card 4111111111111111)"


def decide_route(state):
    route = "rag" if set(re.findall(r"[a-z]+", state["question"].lower())) & POLICY_WORDS else "tool"
    return {"route": route, "trace": [tr(state, "decide_route", "route=" + route)]}


def retrieve(state):
    return {"context": [P1], "trace": [tr(state, "retrieve", "docs=['P1']")]}


def choose_tool(state):
    tool = "lookup_balance" if "balance" in state["question"].lower() else "none"
    return {"tool": tool, "trace": [tr(state, "choose_tool", "tool=" + tool)]}


def run_tool(state):
    if state["tool"] == "none":
        return {"result": "no tool needed", "trace": [tr(state, "run_tool", "none")]}
    return {"result": call_tool(state["cid"], state["tool"], lookup_balance, state["user"]),
            "trace": [tr(state, "run_tool", state["tool"])]}


def write_answer(state):
    if state["fallback"]:   # something upstream failed: no model call, a safe message instead
        return {"answer": "Service busy, please try again later.", "trace": [tr(state, "write_answer", "degraded")]}
    text, tokens = call_model(state["cid"], "Context: " + " ".join(state["context"]) + "\nData: " + state["result"]
                              + "\nQuestion: " + state["question"])
    return {"answer": text, "tokens": tokens, "trace": [tr(state, "write_answer", "tokens=" + str(tokens))]}

print("Workers ready.")

Workers ready.


## Step 4: Build the graph (every worker wrapped in the stopwatch)

```text
START -> decide_route --policy?--> retrieve -> choose_tool -> run_tool -> write_answer -> END
                      \--other-----------------> choose_tool
```

The only new thing is `traced(name, fn)` around each worker. That is all the instrumentation there is.

In [5]:
def after_route(state):
    return "retrieve" if state["route"] == "rag" else "choose_tool"


graph = StateGraph(State)
for node_fn in (decide_route, retrieve, choose_tool, run_tool, write_answer):
    graph.add_node(node_fn.__name__, traced(node_fn.__name__, node_fn))
graph.add_edge(START, "decide_route")
graph.add_conditional_edges("decide_route", after_route, ["retrieve", "choose_tool"])
graph.add_edge("retrieve", "choose_tool")
graph.add_edge("choose_tool", "run_tool")
graph.add_edge("run_tool", "write_answer")
graph.add_edge("write_answer", END)
app = graph.compile()


def ask(question, broken=False):
    """Make ONE tracking number at the front door, run the graph, return (cid, final state)."""
    FAULT["lookup_balance"] = broken
    cid = uuid.uuid4().hex[:8]
    start = {"user": "cust-1", "team": "support", "question": question, "cid": cid, "t0": time.perf_counter(), "route": "",
             "context": [], "tool": "", "result": "", "answer": "", "tokens": 0, "blocked": "", "fallback": False, "trace": []}
    return cid, app.invoke(start)

print("Traced graph built.")

Traced graph built.


## Step 5: Three requests, one black box

Three customers ask three things. All three write into the **same file**. Only the tracking numbers keep them apart.

1. A **policy question** (goes through retrieval, then the model).
2. A **balance question that includes a card number** (tool, then the model). This is our privacy test.
3. The **same balance question, but the tool times out** (we set `broken=True`).

Look at what the *user* sees. Requests 2 and 3 take the same path through the graph, yet one succeeds and one says *"Service busy"*.

In [6]:
PII_QUESTION = "My card is 4111111111111111, what is my balance?"
finals = {}
for question, broken in [("What happens if my card is lost?", False), (PII_QUESTION, False), (PII_QUESTION, True)]:
    cid, final = ask(question, broken)
    finals[cid] = final
    print("User saw (" + cid + "):", final["answer"])

User saw (e549fb28): Here is what I found.
User saw (8f1de776): Here is what I found.
User saw (7d7a7060): Service busy, please try again later.


## Step 6: Open the black box

We read the file back and group the spans by tracking number. For each request we print: the status, the total time of the steps, the tokens, the cost, and the **path** it took.

Read the **path** column first. Requests 2 and 3 have *the same path*. The path alone cannot tell them apart. The *errors* and the *missing model span* can.

In [7]:
def read_spans():
    with open(TRACE_FILE) as trace:
        raw = trace.read()
    return raw, [json.loads(line) for line in raw.splitlines()]


def summarise(records):
    print("Per-request summary")
    for cid in dict.fromkeys(r["cid"] for r in records):   # order of first appearance
        mine = [r for r in records if r["cid"] == cid]
        nodes = [r for r in mine if r["kind"] == "node"]
        tokens = sum(r.get("tokens_in", 0) + r.get("tokens_out", 0) for r in mine)
        cost = sum(r.get("cost_usd", 0) for r in mine)
        status = "DEGRADED" if any(r["error"] for r in mine) else "ok"
        print(" ", cid, status.ljust(8), "total", str(round(sum(r["latency_ms"] for r in nodes), 1)).rjust(6), "ms",
              "tokens", tokens, "cost $" + format(cost, ".6f"), "path", [r["name"] for r in nodes])


raw_trace, records = read_spans()
summarise(records)
print()
print(len(records), "spans in the file for", len({r["cid"] for r in records}), "requests.")

Per-request summary
  e549fb28 ok       total    1.0 ms tokens 51 cost $0.000035 path ['decide_route', 'retrieve', 'choose_tool', 'run_tool', 'write_answer']
  8f1de776 ok       total   54.5 ms tokens 41 cost $0.000029 path ['decide_route', 'choose_tool', 'run_tool', 'write_answer']
  7d7a7060 DEGRADED total   54.5 ms tokens 0 cost $0.000000 path ['decide_route', 'choose_tool', 'run_tool', 'write_answer']

17 spans in the file for 3 requests.


## Step 7: Investigate the failing request

You are the investigator. A customer says *"I got 'Service busy'"*. You have their tracking number.

1. **Read the spans in order** and find the **first error**.
2. **Read the `tr()` lines** the request wrote into its own state. Compare the `+ms` values with the span times.
3. **Write the incident note:** symptom, root cause, evidence.
4. **Run the privacy check:** is the card number anywhere in the trace?

In [8]:
bad_id = [r for r in records if r["error"]][0]["cid"]
failing = [r for r in records if r["cid"] == bad_id]

print("Spans of failing request", bad_id)
for r in failing:
    print(" ", r["kind"].ljust(5), r["name"].ljust(15), str(r["latency_ms"]).rjust(6), "ms", ("ERROR " + r["error"]) if r["error"] else "ok")

print("\nTrace lines in state (same request, tr() format):")
for line in finals[bad_id]["trace"]:
    print(" ", line)

Spans of failing request 7d7a7060
  node  decide_route       0.0 ms ok
  node  choose_tool        0.0 ms ok
  tool  lookup_balance    50.4 ms ERROR TimeoutError: mcp server did not answer in 0.05s
  node  run_tool          54.5 ms ERROR TimeoutError: mcp server did not answer in 0.05s
  node  write_answer       0.0 ms ok

Trace lines in state (same request, tr() format):
  cid=7d7a7060 decide_route +0ms route=tool
  cid=7d7a7060 choose_tool +2ms tool=lookup_balance
  cid=7d7a7060 run_tool +58ms ERROR TimeoutError
  cid=7d7a7060 write_answer +64ms degraded


Notice **two** error lines: the `tool` span (`lookup_balance`) and the `node` span (`run_tool`). They are the same failure seen from two places. The **tool** span is the root cause; the node span is where it was noticed. And `write_answer` shows `ok`. It did its job: it returned the safe fallback. The *symptom* is in `write_answer`, the *cause* is upstream.

In [9]:
first_error = [r for r in failing if r["error"]][0]
slowest = max(failing, key=lambda r: r["latency_ms"])
final = finals[bad_id]
print("Incident note")
print("  Symptom   : answer was 'Service busy' (fallback=" + str(final["fallback"]) + ")")
print("  Root cause: first error is the", first_error["kind"], "'" + first_error["name"] + "' -> " + first_error["error"])
print("  Evidence  : slowest span", slowest["name"], str(slowest["latency_ms"]) + " ms; model spans in this request:",
      len([r for r in failing if r["kind"] == "model"]))
print("  PII check : card number in trace file?", "4111111111111111" in raw_trace, "| in state trace?", "4111111111111111" in str(final["trace"]))

Incident note
  Symptom   : answer was 'Service busy' (fallback=True)
  Root cause: first error is the tool 'lookup_balance' -> TimeoutError: mcp server did not answer in 0.05s
  Evidence  : slowest span run_tool 54.5 ms; model spans in this request: 0
  PII check : card number in trace file? False | in state trace? False


### What the output proves

| Line | What it proves | Idea |
| --- | --- | --- |
| Same path for requests 2 and 3 | The path alone is not enough | Spans carry the errors |
| `tool lookup_balance ... ERROR` first | The root cause is the first error span | Root cause vs symptom |
| `model spans in this request: 0` | The fallback skipped the model, so no tokens and no cost | Missing span is evidence |
| `+60ms`-style numbers | Time is cumulative from `t0` | `tr()` format |
| `PII check` prints False twice | The card number never reached the file or the log lines | Privacy |

## Step 8: Your turn. Predict first, then run

### Experiment 1: a slow tool that does not fail

We switch the fault **off** and make the balance tool take **1 second**. **Will there be any error spans? Which spans will show the delay?**

<details><summary><b>Click to see the answer</b></summary>

**No errors**, but the `lookup_balance` tool span and the `run_tool` node span both show about **1000 ms**, and they dominate the total. The field that tells you is `latency_ms`. A slow step is a problem even when nothing "fails".
</details>

In [10]:
TOOL["delay_s"] = 1.0
cid, final = ask(PII_QUESTION, broken=False)
_, records = read_spans()
for r in [r for r in records if r["cid"] == cid]:
    print(" ", r["kind"].ljust(5), r["name"].ljust(15), str(r["latency_ms"]).rjust(7), "ms", ("ERROR " + r["error"]) if r["error"] else "ok")
TOOL["delay_s"] = 0.05   # put it back

  node  decide_route        0.0 ms ok
  node  choose_tool         0.0 ms ok
  tool  lookup_balance   1000.3 ms ok
  node  run_tool         1001.3 ms ok
  model gpt-oss:120b-cloud     0.0 ms ok
  node  write_answer        0.9 ms ok


### Experiment 2: what if we logged the question?

A careless colleague adds the *question text* to a span. **Will the privacy check catch it?**

<details><summary><b>Click to see the answer</b></summary>

**Yes: the check prints `True`.** That is why we keep a test that searches the trace for the card number. Spans must hold sizes, counts and names, never the raw text. A privacy rule that nobody tests quietly breaks.
</details>

In [11]:
leaky = json.dumps({"cid": "demo", "kind": "node", "name": "decide_route", "question": PII_QUESTION})   # the careless version
print("Card number in the careless span?", "4111111111111111" in leaky)

safe = json.dumps({"cid": "demo", "kind": "node", "name": "decide_route", "question_chars": len(PII_QUESTION)})   # the careful version
print("Card number in the careful span? ", "4111111111111111" in safe)

Card number in the careless span? True
Card number in the careful span?  False


### Experiment 3: a tracking number per step (the common mistake)

Suppose each step made **its own** new `cid` instead of using the one from the front door. **How many different tracking numbers would one request that runs 5 steps have, and can you still find "its" spans?**

<details><summary><b>Click to see the answer</b></summary>

**5 different numbers**, so you could not group the spans into one request. Make the `cid` **once per request, at the edge**, and carry it in the state.
</details>

In [12]:
per_step = [uuid.uuid4().hex[:8] for _ in range(5)]   # the wrong way: a new id inside every step
print("Different tracking numbers for 1 request:", len(set(per_step)))
print("Right way (one id from the front door):  ", len({r["cid"] for r in failing}), "(the failing request's real spans)")

Different tracking numbers for 1 request: 5
Right way (one id from the front door):   1 (the failing request's real spans)


### Experiment 4: the node and the tool, who owns the time?

In the failing request, `run_tool` took about 50 ms more than nothing and `lookup_balance` almost the same. **Is the time spent in the node's own code or in the tool?**

<details><summary><b>Click to see the answer</b></summary>

**In the tool.** The tool span is almost as long as the node span around it, so only a few ms are the node's own work. Nesting tells you where the time went.
</details>

In [13]:
node_span = [r for r in failing if r["name"] == "run_tool"][0]
tool_span = [r for r in failing if r["kind"] == "tool"][0]
print("run_tool node :", node_span["latency_ms"], "ms")
print("lookup_balance:", tool_span["latency_ms"], "ms")
print("The node's own code took about", round(node_span["latency_ms"] - tool_span["latency_ms"], 1), "ms")

run_tool node : 54.5 ms
lookup_balance: 50.4 ms
The node's own code took about 4.1 ms


### Tidy up

Delete the black box file so the notebook leaves no mess.

In [14]:
if os.path.exists(TRACE_FILE):
    os.remove(TRACE_FILE)
print("Trace file removed:", not os.path.exists(TRACE_FILE))

Trace file removed: True


## Step 9: Assignment (instrument the agent and diagnose a deliberately introduced failure)

1. **Instrument your own graph** (the Day 3 SupportDesk refactor, or your L2 agent) with `traced`, `tr()` lines, a tool span and a model span.
2. A **partner changes one line** to inject a failure (a slow tool, an exception, or empty context) without telling you.
3. From the **trace only**, write a four-line **incident note**: symptom, root-cause span, evidence (the JSON line), impact (how many requests, how much extra latency or cost).
4. **Show that the trace file and the `trace` lines contain no personal data** (use the question "My card is 4111111111111111, what is my balance?").

## Production checklist (you will reuse these lines on Day 19)
- [ ] **Every node, tool call and model call** emits a span with `cid`, `latency_ms` and `error`; a test counts the spans of one request.
- [ ] **Model spans** carry real `tokens_in`, `tokens_out` and `cost_usd`, and per-request totals are computed from them.
- [ ] **One `cid` per request**, created at the API edge, on every span and every `tr()` line (and later in the audit log).
- [ ] **A test searches the trace** for the card number, ID or secret used in the request and finds nothing.
- [ ] **Someone other than the author** can name the first failing span from the trace alone.

## Recap in one minute
| Idea | The analogy | One-line takeaway |
| --- | --- | --- |
| Span | One scan of the parcel | One JSON line: who, what, how long, did it fail. |
| Correlation id | The tracking number | Made once at the front door, on every line. |
| node / tool / model | Sorting centre, truck, AI | A tool span inside a node shows where the time went. |
| Tokens and cost | Fuel used | Real counts on every model span, totalled per request. |
| Wrapper | Stopwatch and camera | Instrument without touching the steps. |
| Privacy | The recorder never stores the conversation | Log sizes and names, and test for leaks. |
| Incident note | The investigator's report | The symptom is last; the cause is the first error. |

## Tomorrow (Day 5)
Now that you can *see* a failure, you can *harden* against it: timeouts on every call, bounded retries, a fallback path, and failure injection that you check with today's trace.


## Step 10 (optional): the same trail in a tracing tool

Everything above is a tiny tracing tool made of JSON lines. Hosted tools (LangSmith, Langfuse, Phoenix and others) store, show and search the same facts. With **LangSmith** you set a few variables in `.env` and add one decorator:

```text
LANGCHAIN_TRACING_V2=true
LANGCHAIN_API_KEY=lsv2_...
LANGCHAIN_PROJECT=l3-day04
```

`@traceable` makes any plain function show up as its own span (just like our `call_tool`). `tags` and `metadata` let you filter runs later (like our `cid`).

**No key? No problem.** This cell skips itself and the course does not need it. Keep keys in `.env` and keep `.env` in `.gitignore`.

In [1]:
import os

from dotenv import load_dotenv
from langsmith import traceable

load_dotenv()
has_key = bool(os.getenv("LANGCHAIN_API_KEY") or os.getenv("LANGSMITH_API_KEY"))
is_on = os.getenv("LANGCHAIN_TRACING_V2", "").lower() == "true" or os.getenv("LANGSMITH_TRACING", "").lower() == "true"


@traceable(name="check_refund_eligibility", run_type="tool")   # a plain function becomes a span
def check_eligibility(ticket):
    return "damaged" in ticket.lower()


if has_key and is_on:
    result = check_eligibility("My package arrived damaged", langsmith_extra={"tags": ["day04"], "metadata": {"cohort": "L3"}})
    print("Eligible:", result, "-> open your LangSmith project and find the run tagged day04")
else:
    print("Skipped: no LangSmith key and tracing switch found in .env. That is fine.")

Skipped: no LangSmith key and tracing switch found in .env. That is fine.
